# ResNet18 on CIFAR-10

这个 Notebook 从 `CIFAR-10` 数据集加载开始，完整展示 `Resize(224x224) -> ResNet18` 的训练与结构分析流程。

内容包括：
- 数据集预处理与可视化
- `DataLoader` 构建
- ResNet18 与残差块实现
- 逐层尺寸变化分析
- 残差连接作用解读
- 训练、验证与预测展示

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# 数学工具用于后面的可视化排版等辅助逻辑
import math
# dataclass 用于统一管理实验配置
from dataclasses import dataclass

# matplotlib 用于图像和训练曲线可视化
import matplotlib.pyplot as plt
# PyTorch 核心模块
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
# torchvision 提供数据集和图像预处理工具
from torchvision import datasets, transforms

# 设置绘图风格，便于 Notebook 展示
plt.style.use('seaborn-v0_8')
# 固定随机种子，方便复现
torch.manual_seed(42)

# 优先使用 GPU，没有则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据下载与缓存目录
    data_root: str = './data'
    # 将 CIFAR-10 从 32x32 拉伸到 224x224
    image_size: int = 224
    # 每个 batch 的样本数
    batch_size: int = 64
    # DataLoader 并行加载进程数
    num_workers: int = 2
    # 学习率
    lr: float = 1e-3
    # 训练轮数
    epochs: int = 5


cfg = Config()
cfg

## 2. 加载 CIFAR-10 并拉伸到 224x224

ResNet18 常用于较大尺寸输入场景。这里把 `CIFAR-10` 统一拉伸到 `224x224`，便于观察残差网络的下采样路径和阶段式结构。

In [ ]:
# CIFAR-10 常用归一化参数
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# 训练集：尺寸拉伸、随机翻转、张量化、归一化
train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 测试集不做随机增强，只保留基础预处理
test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 本地不存在数据时自动下载
train_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=False,
    download=True,
    transform=test_transform,
)

# CIFAR-10 类别名称
classes = train_dataset.classes
classes

In [ ]:
def denormalize(image_tensor, mean, std):
    # 反归一化，便于可视化时恢复到更自然的像素范围
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


# 展示若干样本，确认输入图像尺寸与视觉效果
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), range(8)):
    image, label = train_dataset[idx]
    image = denormalize(image, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis('off')

plt.suptitle('CIFAR-10 samples resized to 224x224', fontsize=16)
plt.tight_layout()
plt.show()

## 3. 构建 DataLoader

In [ ]:
# 训练集打乱顺序，减少模型对样本顺序的依赖
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    # GPU 训练时开启 pin_memory 往往更高效
    pin_memory=torch.cuda.is_available(),
)

# 测试集保持固定顺序即可
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 检查一个 batch 的维度
images, labels = next(iter(train_loader))
print('batch image shape:', images.shape)
print('batch label shape:', labels.shape)

## 4. ResNet18 实现

ResNet 的核心思想是残差连接。与其让网络直接学习完整映射，不如让网络学习“输入和输出之间的残差”，这样更容易训练更深的模型。

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # 主分支第一层卷积，必要时通过 stride 做下采样
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        # 主分支第二层卷积，用于进一步提取特征
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 如果输入输出形状不同，需要用捷径分支做维度对齐
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 残差连接：主分支输出与捷径分支相加
        out += identity
        out = self.relu(out)
        return out


class ResNet18CIFAR10(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            # 首层使用较大卷积核，快速提取低层视觉模式
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        # 四个阶段，每个阶段包含两个 BasicBlock
        self.layer1 = self._make_layer(64, 64, blocks=2, stride=1)
        self.layer2 = self._make_layer(64, 128, blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, blocks=2, stride=2)
        # 全局平均池化把每个通道压成一个标量
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # 最终分类层输出 10 个类别
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = [BasicBlock(in_channels, out_channels, stride=stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


model = ResNet18CIFAR10().to(device)
model

## 5. 逐层尺寸变化分析

In [ ]:
def inspect_feature_shapes(model, input_shape=(1, 3, 224, 224)):
    # 构造一个假的输入，只用于追踪每个阶段的尺寸变化
    x = torch.randn(input_shape)
    print(f'input: {tuple(x.shape)}')

    x = model.stem(x)
    print(f'stem   -> {tuple(x.shape)}')

    x = model.layer1(x)
    print(f'layer1 -> {tuple(x.shape)}')
    x = model.layer2(x)
    print(f'layer2 -> {tuple(x.shape)}')
    x = model.layer3(x)
    print(f'layer3 -> {tuple(x.shape)}')
    x = model.layer4(x)
    print(f'layer4 -> {tuple(x.shape)}')

    x = model.avgpool(x)
    print(f'avgpool -> {tuple(x.shape)}')

    x = torch.flatten(x, 1)
    print(f'flatten -> {tuple(x.shape)}')

    x = model.fc(x)
    print(f'fc      -> {tuple(x.shape)}')


inspect_feature_shapes(model.cpu())
model = model.to(device)

## 6. 残差连接为什么重要？

1. 深层网络优化更稳定
   - 残差连接为梯度传播提供了更直接的路径。

2. 模型更容易学习恒等映射
   - 如果某些层暂时不需要复杂变换，网络更容易保留输入信息。

3. 更深但不一定更难训练
   - 这也是 ResNet 相比 AlexNet、VGG16 的关键突破之一。

## 7. 参数量统计

In [ ]:
def count_parameters(model):
    # 只统计可训练参数
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


total_params = count_parameters(model)
print(f'Trainable parameters: {total_params:,}')

## 8. 训练与验证函数

In [ ]:
# 多分类任务常用交叉熵损失
criterion = nn.CrossEntropyLoss()
# 使用 Adam 便于快速开始实验
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # 训练模式会启用 BatchNorm 的训练行为
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        # 把一个 batch 的数据放到目标设备上
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    # 评估模式关闭训练期随机行为
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total

## 9. 训练主循环

默认训练 `5` 个 epoch。由于输入是 `224x224`，如果在 CPU 上运行会比较慢。

In [ ]:
# 记录训练和验证指标，后面用于画曲线
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

# 每轮先训练，再评估
for epoch in range(cfg.epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

In [ ]:
# 绘制损失曲线和准确率曲线
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], label='train loss')
axes[0].plot(epochs, history['val_loss'], label='val loss')
axes[0].set_title('Loss curves')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='train acc')
axes[1].plot(epochs, history['val_acc'], label='val acc')
axes[1].set_title('Accuracy curves')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. 预测结果展示

In [ ]:
@torch.no_grad()
def show_predictions(model, dataloader, class_names, device, num_images=8):
    # 推理前先切换到评估模式
    model.eval()
    # 取一个 batch 做展示
    images, labels = next(iter(dataloader))
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    preds = logits.argmax(dim=1)

    fig, axes = plt.subplots(2, math.ceil(num_images / 2), figsize=(16, 6))
    axes = axes.flatten()

    for i in range(num_images):
        # 反归一化后再显示图像
        image = denormalize(images[i].cpu(), cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
        axes[i].imshow(image)
        axes[i].set_title(f'true: {class_names[labels[i]]}\npred: {class_names[preds[i]]}')
        axes[i].axis('off')

    for i in range(num_images, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, classes, device)